# L2 latency bench — Kaggle (offline / pure-run)

Reads everything from the `parapet-l2-bench` Kaggle dataset (wheel, ONNX model, tokenizer, corpus, hashes, revision). No HF Hub fetches, no `optimum-cli`, no on-Kaggle export — that all happened upstream in `build_l2_latency_bundle.ipynb`.

**Runtime**: CPU. `direction.md` Phase 0.6 gates: `end_to_end.p50_ms ≤ 25`, `end_to_end.p99_ms ≤ 100`.

**One placeholder**: if you uploaded the dataset under a slug other than `parapet-l2-bench`, search/replace it across cells 2–4.

In [ ]:
# Install parapet-runner with the [bench] extra from the wheel mounted in the
# Kaggle dataset. PEP 508 file:// form so pip resolves the [bench] extra cleanly.
# Internet must be on so pip can fetch any [bench] deps not already in Kaggle's
# base image (onnxruntime / transformers / sentencepiece / pydantic / pyyaml /
# numpy). Anything already present is a no-op.
!pip install "parapet-runner[bench] @ file:///kaggle/input/parapet-l2-bench/parapet_runner-0.1.0-py3-none-any.whl"

In [ ]:
# All bench logic lives in parapet_runner.latency_bench. Hashes and revision
# are pre-computed in the bundle; we read them at the shell level.
!python -m parapet_runner.latency_bench \
    --model-path /kaggle/input/parapet-l2-bench/model/model.int8.onnx \
    --tokenizer-path /kaggle/input/parapet-l2-bench/model \
    --model-revision microsoft/mdeberta-v3-base@$(cat /kaggle/input/parapet-l2-bench/model/revision.txt) \
    --onnx-sha256 $(cat /kaggle/input/parapet-l2-bench/model/model.int8.sha256) \
    --quant int8 \
    --provider CPUExecutionProvider \
    --corpus /kaggle/input/parapet-l2-bench/corpus/l2_latency_v8_train_val_stratified.jsonl \
    --output /kaggle/working/latency_result.json \
    --environment kaggle

In [ ]:
import json
result = json.loads(open("/kaggle/working/latency_result.json").read())
print("end_to_end (ms):", result["end_to_end"])
print("tokenize   (ms):", result["tokenize"])
print("infer      (ms):", result["infer"])
print("token length histogram:", result["token_length_histogram"])
print("\nmanifest:")
print(json.dumps(result["manifest"], indent=2))
print("\n--- gate check (direction.md Phase 0.6) ---")
p50 = result["end_to_end"]["p50_ms"]
p99 = result["end_to_end"]["p99_ms"]
print(f"p50 = {p50:.2f} ms (gate: \u2264 25)  {'PASS' if p50 <= 25 else 'FAIL'}")
print(f"p99 = {p99:.2f} ms (gate: \u2264 100) {'PASS' if p99 <= 100 else 'FAIL'}")